#  Data Cleaning

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

raw_counts = {
    "orders": len(orders),
    "order_items": len(items),
    "payments": len(payments),
    "customers": len(customers),
    "products": len(products),
    "sellers": len(sellers),
}

for name, n in raw_counts.items():
    print(f"{name:12} raw rows: {n:>8,}")

orders       raw rows:   99,441
order_items  raw rows:  112,650
payments     raw rows:  103,886
customers    raw rows:   99,441
products     raw rows:   32,951
sellers      raw rows:    3,095


# Change the Datatypes the Date_Coloumn

In [3]:
orders_clean = orders.copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean[date_cols] = orders_clean[date_cols].apply(
    pd.to_datetime,
    errors="coerce"
)

print(orders_clean[date_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


# to fill the missing value of the product category

In [4]:
# Create a copy so the original products data stays unchanged
products_clean = products.copy()

# Check existing "unknown" values and missing values before cleaning
print("Existing 'unknown' values:",
      (products_clean["product_category_name"] == "unknown").sum())

print("Missing categories before cleaning:",
      products_clean["product_category_name"].isna().sum())


# Replace only actual missing values (NaN) with "unknown"
products_clean["product_category_name"] = (
    products_clean["product_category_name"].fillna("unknown")
)


# Check after cleaning
print("\nExisting/missing category check after cleaning:")
print("Missing categories after cleaning:",
      products_clean["product_category_name"].isna().sum())

print("Total 'unknown' values after cleaning:",
      (products_clean["product_category_name"] == "unknown").sum())

Existing 'unknown' values: 0
Missing categories before cleaning: 610

Existing/missing category check after cleaning:
Missing categories after cleaning: 0
Total 'unknown' values after cleaning: 610


# Category Names and Handle Missing Translations

In [5]:
# Add English category names
products_clean = products_clean.merge(
    translation,
    on="product_category_name",
    how="left"
)

# Handle missing English translations
products_clean["product_category_name_english"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"])
)

# Check the result
print(
    "Missing English category names:",
    products_clean["product_category_name_english"].isna().sum()
)

Missing English category names: 0


#  Validate Monetary Values

In [6]:
print("Negative payment values:",
      (payments_clean["payment_value"] < 0).sum())

print("Zero payment values:",
      (payments_clean["payment_value"] == 0).sum())

print("Negative item prices:",
      (items_clean["price"] < 0).sum())

print("Zero item prices:",
      (items_clean["price"] == 0).sum())

print("Negative freight values:",
      (items_clean["freight_value"] < 0).sum())

print("Zero freight values:",
      (items_clean["freight_value"] == 0).sum())

NameError: name 'payments_clean' is not defined